# 00A｜HiDF 資料集抽樣與切分

建立 HiDF 的 train／val／test 資料夾。

**本 Notebook 屬於作者先前的專題研究**，原用途是為舊版模型準備訓練資料。在本次研究中，**只有其產生的 val 子集（Real 313、Fake 313，共 626 張）被取出，作為外部測試集使用**。本次的模型完全在 FF++ 上訓練，未接觸 HiDF 的任何資料。

保留這份 Notebook 是為了說明 04 與 08 所用的 626 張 HiDF 影像從何而來。

**處理方式**

- 讀取 `metadata.csv`，取得每個 ID 對應的種族標註。
- 將每個種族的 ID 隨機均分為兩半：一半只允許出現在 Real、另一半只允許出現在 Fake，降低同一身分同時以真、假兩種形式出現的風險。
- 依檔名規則分流：Real 檔名無底線（如 `00001.jpg`），Fake 檔名含底線（如 `00001_00002.jpg`，前兩段分別為 face ID 與 body ID）。Fake 只保留 face 與 body 種族相同者。
- 各種族盡量平均抽樣，依 8:1:1 切為 train／val／test。
- 不做人臉裁切，只複製原始圖片。

**三項已知限制（影響本次研究對 HiDF 結果的解讀）**

**一、Fake 的種族分布嚴重偏斜。** 抽樣後 Real 各族接近均衡（White/Black/Asian/Latino 各約 690、Indian 353），但 Fake 有 2,427 張為 White，佔 78%（Black 236、Asian 358、Latino 90、Indian 15）。原因是 Fake 的可用圖片庫存本身即高度不均（White 5,126 對 Indian 15），平均抽樣無法補足。這代表模型有可能部分依賴與種族相關的特徵區分 Real／Fake，而非偽造痕跡本身。

**二、切分不以身分為單位。** train／val／test 是將圖片路徑打散後依數量切分，同一個 ID 可能同時出現在三個子集中。對本次研究無影響（模型從未接觸 HiDF 的任何子集），但若以此 val 集評估曾在 HiDF train 上訓練過的模型，成績會因身分洩漏而偏高。

**三、未做人臉裁切。** 本次研究的訓練資料（FF++）經過 YOLO 正方形裁切，HiDF 則直接使用原始圖片。前處理與訓練時不一致，此差異在解讀 HiDF 結果時應一併考量。

In [1]:
# ==========================================
#  參數設定區
# ==========================================

# metadata.csv 路徑
CSV_FILE_PATH = "../../../HiDF/metadata.csv"

# 原始圖片資料夾路徑
SOURCE_DATASET_FOLDER = "../../../HiDF"

# 輸出資料夾
TARGET_OUTPUT_DIR = "dataset_HiDF5000"

# 這裡代表「train 資料夾裡總共幾張圖片」
# 例如 1000 = train/real 500 + train/fake 500
TRAIN_COUNT_TOTAL = 5000

# 切分比例：train : val : test = 8 : 1 : 1
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# 要保留並平衡抽樣的人種欄位
VALID_RACES_LIST = ["White", "Black", "Asian", "Latino", "Indian"]

# 固定種子，讓每次抽樣結果可重現
# 若想每次隨機不同，可改成 None
SEED = 42

# 如果輸出資料夾已存在，是否先刪掉重建
RESET_OUTPUT_DIR = True


In [2]:
import os
import math
import random
import shutil
from collections import Counter
from pathlib import Path

import pandas as pd
from tqdm import tqdm


IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def set_seed(seed=None):
    if seed is not None:
        random.seed(seed)


def load_metadata(csv_path):
    """
    讀取 metadata.csv，自動找出 ID 欄位與 race 欄位。
    回傳：
        id_race_map = {"001": "Asian", "002": "White", ...}
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"找不到 CSV：{csv_path}")

    df = pd.read_csv(csv_path)

    id_col = None
    race_col = None

    for col in df.columns:
        lower_col = col.lower().strip()

        if id_col is None and (
            lower_col == "id"
            or "person_id" in lower_col
            or lower_col.endswith("id")
        ):
            id_col = col

        if race_col is None and (
            "race" in lower_col
            or "ethnicity" in lower_col
        ):
            race_col = col

    if id_col is None or race_col is None:
        raise ValueError(
            "CSV 欄位識別失敗，找不到 ID 或 race 欄位。\n"
            f"目前欄位：{list(df.columns)}"
        )

    id_race_map = {}
    for _, row in df.iterrows():
        pid = str(row[id_col]).strip()
        race = str(row[race_col]).strip()

        if pid and pid.lower() != "nan" and race and race.lower() != "nan":
            id_race_map[pid] = race

    print(f"metadata 載入完成：{len(id_race_map)} 個 ID")
    print(f"使用 ID 欄位：{id_col}")
    print(f"使用 race 欄位：{race_col}")

    return id_race_map


def partition_ids_by_race(id_race_map, valid_races):
    """
    將每個 race 的 ID 隨機切成兩半：
    - real_allowed：只允許出現在 real 圖片
    - fake_allowed：只允許出現在 fake 圖片

    目的：降低同一個人同時出現在 real/fake 的資料洩漏風險。
    """
    race_groups = {race: [] for race in valid_races}

    for pid, race in id_race_map.items():
        if race in race_groups:
            race_groups[race].append(pid)

    real_allowed = set()
    fake_allowed = set()

    print("\n各 race 的 ID 數量：")
    for race, ids in race_groups.items():
        random.shuffle(ids)
        mid = len(ids) // 2

        real_ids = ids[:mid]
        fake_ids = ids[mid:]

        real_allowed.update(real_ids)
        fake_allowed.update(fake_ids)

        print(f"  {race:<8} total={len(ids):>5} | real_allowed={len(real_ids):>5} | fake_allowed={len(fake_ids):>5}")

    print(f"\nID 分流完成：real_allowed={len(real_allowed)}, fake_allowed={len(fake_allowed)}")
    return real_allowed, fake_allowed


def scan_and_filter_strict(dataset_dir, id_race_map, real_allowed, fake_allowed, valid_races):
    """
    掃描原始圖片資料夾，依檔名規則分成 real/fake buckets。

    原本 notebook 的判斷邏輯：
    - real 圖片檔名：沒有底線，例如 00001.jpg，檔名本體視為 person ID。
    - fake 圖片檔名：有底線，例如 00001_00002.jpg，前兩段視為 fake 的 face ID / body ID。
    - fake 只保留 face ID 和 body ID 的 race 相同者。
    """
    real_buckets = {race: [] for race in valid_races}
    fake_buckets = {race: [] for race in valid_races}

    dataset_dir = os.path.abspath(dataset_dir)
    if not os.path.exists(dataset_dir):
        raise FileNotFoundError(f"找不到原始圖片資料夾：{dataset_dir}")

    print(f"\n開始掃描資料夾：{dataset_dir}")

    total_images = 0
    used_real = 0
    used_fake = 0

    for root, _, files in os.walk(dataset_dir):
        for filename in files:
            if not filename.lower().endswith(IMAGE_EXTS):
                continue

            total_images += 1
            name_no_ext = os.path.splitext(filename)[0]
            full_path = os.path.join(root, filename)

            if "_" in name_no_ext:
                parts = name_no_ext.split("_")
                if len(parts) < 2:
                    continue

                fid = parts[0]
                bid = parts[1]

                if fid not in id_race_map or bid not in id_race_map:
                    continue

                if fid not in fake_allowed or bid not in fake_allowed:
                    continue

                face_race = id_race_map[fid]
                body_race = id_race_map[bid]

                if face_race == body_race and face_race in valid_races:
                    fake_buckets[face_race].append(full_path)
                    used_fake += 1

            else:
                pid = name_no_ext

                if pid in id_race_map and pid in real_allowed:
                    race = id_race_map[pid]
                    if race in valid_races:
                        real_buckets[race].append(full_path)
                        used_real += 1

    print(f"掃描圖片總數：{total_images}")
    print(f"可用 real 圖片：{used_real}")
    print(f"可用 fake 圖片：{used_fake}")

    return real_buckets, fake_buckets


def print_bucket_summary(title, buckets):
    print(f"\n{title}")
    total = 0
    for race, paths in buckets.items():
        total += len(paths)
        print(f"  {race:<8}: {len(paths):>6}")
    print(f"  {'TOTAL':<8}: {total:>6}")


def balanced_sample(buckets, target_total):
    """
    從各 race bucket 盡量平均抽樣，直到達到 target_total。
    若某些 race 圖片不足，會把剩餘需求分配給還有圖片的 race。
    """
    if target_total <= 0:
        return []

    pool = {race: paths[:] for race, paths in buckets.items()}
    for race in pool:
        random.shuffle(pool[race])

    selected = []
    remain = target_total
    races = [race for race, paths in pool.items() if len(paths) > 0]

    while remain > 0 and races:
        quota = math.ceil(remain / len(races))
        next_races = []

        for race in races:
            paths = pool[race]
            take = min(len(paths), quota, remain)

            selected.extend(paths[:take])
            pool[race] = paths[take:]
            remain -= take

            if len(pool[race]) > 0:
                next_races.append(race)

            if remain <= 0:
                break

        races = next_races

    if len(selected) < target_total:
        print(f"[警告] 目標需要 {target_total} 張，但可抽到的圖片只有 {len(selected)} 張。")

    random.shuffle(selected)
    return selected


def count_selected_by_race(buckets, selected_paths):
    selected_set = set(os.path.abspath(p) for p in selected_paths)
    counter = Counter()

    for race, paths in buckets.items():
        for path in paths:
            if os.path.abspath(path) in selected_set:
                counter[race] += 1

    return counter


def calculate_split_counts(train_count_total, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """
    用 train_count_total 控制 train 總張數。

    例如：
        train_count_total = 1000
        train_ratio = 0.8

    則：
        train/real = 500
        train/fake = 500
        val/test 依 8:1:1 推回，約每類 63 張。

    為了維持 real/fake 平衡，val/test 會使用每類整數張數，總比例會非常接近 8:1:1。
    """
    if train_count_total <= 0:
        raise ValueError("TRAIN_COUNT_TOTAL 必須大於 0。")

    ratio_sum = train_ratio + val_ratio + test_ratio
    if abs(ratio_sum - 1.0) > 1e-6:
        raise ValueError(f"TRAIN_RATIO + VAL_RATIO + TEST_RATIO 必須等於 1，目前是 {ratio_sum}")

    if train_count_total % 2 != 0:
        raise ValueError("TRAIN_COUNT_TOTAL 建議設定為偶數，才能讓 real/fake 訓練數量相同。")

    train_per_class = train_count_total // 2

    val_per_class = math.ceil(train_per_class * val_ratio / train_ratio)
    test_per_class = math.ceil(train_per_class * test_ratio / train_ratio)

    counts = {
        "train": train_per_class,
        "val": val_per_class,
        "test": test_per_class,
    }

    total_per_class = sum(counts.values())

    print("\n資料切分數量規劃：")
    print(f"  train/real = {counts['train']}")
    print(f"  train/fake = {counts['train']}")
    print(f"  val/real   = {counts['val']}")
    print(f"  val/fake   = {counts['val']}")
    print(f"  test/real  = {counts['test']}")
    print(f"  test/fake  = {counts['test']}")
    print(f"  train total = {counts['train'] * 2}")
    print(f"  val total   = {counts['val'] * 2}")
    print(f"  test total  = {counts['test'] * 2}")
    print(f"  all total   = {total_per_class * 2}")

    return counts


def split_by_counts(paths, counts):
    """
    將某一類別，例如 real 或 fake，依 counts 切成 train/val/test。
    """
    need_total = counts["train"] + counts["val"] + counts["test"]

    if len(paths) < need_total:
        raise ValueError(
            f"圖片數量不足：需要 {need_total} 張，但只有 {len(paths)} 張。"
        )

    paths = paths[:]
    random.shuffle(paths)

    n_train = counts["train"]
    n_val = counts["val"]
    n_test = counts["test"]

    return {
        "train": paths[:n_train],
        "val": paths[n_train:n_train + n_val],
        "test": paths[n_train + n_val:n_train + n_val + n_test],
    }


def make_unique_output_path(save_dir, filename):
    """
    避免不同資料夾中同名圖片複製到同一個位置時互相覆蓋。
    """
    save_dir = Path(save_dir)
    stem = Path(filename).stem
    suffix = Path(filename).suffix

    out_path = save_dir / filename
    if not out_path.exists():
        return out_path

    idx = 1
    while True:
        candidate = save_dir / f"{stem}_dup{idx}{suffix}"
        if not candidate.exists():
            return candidate
        idx += 1


def copy_split_files(split_dict, target_root, category):
    """
    將 split_dict 裡的圖片複製到：
        target_root/train/category
        target_root/val/category
        target_root/test/category
    """
    copied = {"train": 0, "val": 0, "test": 0}

    for split_name, paths in split_dict.items():
        save_dir = Path(target_root) / split_name / category
        save_dir.mkdir(parents=True, exist_ok=True)

        for src_path in tqdm(paths, desc=f"copy {split_name}/{category}", leave=False):
            src_path = Path(src_path)
            dst_path = make_unique_output_path(save_dir, src_path.name)
            shutil.copy2(src_path, dst_path)
            copied[split_name] += 1

    return copied


def print_output_summary(target_root):
    print("\n輸出資料夾數量檢查：")

    total = 0
    for split_name in ["train", "val", "test"]:
        for category in ["real", "fake"]:
            folder = Path(target_root) / split_name / category
            count = 0
            if folder.exists():
                count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

            total += count
            print(f"  {split_name:<5}/{category:<4}: {count:>6}")

    print(f"  {'TOTAL':<10}: {total:>6}")


def run_prepare_dataset(
    csv_path,
    source_dataset_dir,
    target_root,
    train_count_total,
    valid_races,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
    reset_output_dir=True,
):
    set_seed(seed)

    if reset_output_dir and os.path.exists(target_root):
        print(f"刪除舊資料夾：{target_root}")
        shutil.rmtree(target_root)

    Path(target_root).mkdir(parents=True, exist_ok=True)

    id_race_map = load_metadata(csv_path)

    real_allowed, fake_allowed = partition_ids_by_race(
        id_race_map=id_race_map,
        valid_races=valid_races,
    )

    real_buckets, fake_buckets = scan_and_filter_strict(
        dataset_dir=source_dataset_dir,
        id_race_map=id_race_map,
        real_allowed=real_allowed,
        fake_allowed=fake_allowed,
        valid_races=valid_races,
    )

    print_bucket_summary("Real 可用圖片庫存：", real_buckets)
    print_bucket_summary("Fake 可用圖片庫存：", fake_buckets)

    counts = calculate_split_counts(
        train_count_total=train_count_total,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
    )

    total_per_class = counts["train"] + counts["val"] + counts["test"]

    selected_real = balanced_sample(real_buckets, total_per_class)
    selected_fake = balanced_sample(fake_buckets, total_per_class)

    if len(selected_real) < total_per_class:
        raise ValueError(f"Real 圖片不足，需要 {total_per_class} 張，但只抽到 {len(selected_real)} 張。")

    if len(selected_fake) < total_per_class:
        raise ValueError(f"Fake 圖片不足，需要 {total_per_class} 張，但只抽到 {len(selected_fake)} 張。")

    print("\n抽樣後 race 分布：")
    real_race_count = count_selected_by_race(real_buckets, selected_real)
    fake_race_count = count_selected_by_race(fake_buckets, selected_fake)

    for race in valid_races:
        print(f"  {race:<8} real={real_race_count[race]:>5} | fake={fake_race_count[race]:>5}")

    real_split = split_by_counts(selected_real, counts)
    fake_split = split_by_counts(selected_fake, counts)

    copy_split_files(real_split, target_root, "real")
    copy_split_files(fake_split, target_root, "fake")

    print_output_summary(target_root)
    print("\n資料集建立完成。")


In [3]:
run_prepare_dataset(
    csv_path=CSV_FILE_PATH,
    source_dataset_dir=SOURCE_DATASET_FOLDER,
    target_root=TARGET_OUTPUT_DIR,
    train_count_total=TRAIN_COUNT_TOTAL,
    valid_races=VALID_RACES_LIST,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    reset_output_dir=RESET_OUTPUT_DIR,
)


metadata 載入完成：40361 個 ID
使用 ID 欄位：ID
使用 race 欄位：Race

各 race 的 ID 數量：
  White    total=30348 | real_allowed=15174 | fake_allowed=15174
  Black    total= 2469 | real_allowed= 1234 | fake_allowed= 1235
  Asian    total= 3729 | real_allowed= 1864 | fake_allowed= 1865
  Latino   total= 3051 | real_allowed= 1525 | fake_allowed= 1526
  Indian   total=  764 | real_allowed=  382 | fake_allowed=  382

ID 分流完成：real_allowed=20179, fake_allowed=20182

開始掃描資料夾：D:\資料\專題\HiDF
掃描圖片總數：69831
可用 real 圖片：19264
可用 fake 圖片：5825

Real 可用圖片庫存：
  White   :  14496
  Black   :   1136
  Asian   :   1798
  Latino  :   1481
  Indian  :    353
  TOTAL   :  19264

Fake 可用圖片庫存：
  White   :   5126
  Black   :    236
  Asian   :    358
  Latino  :     90
  Indian  :     15
  TOTAL   :   5825

資料切分數量規劃：
  train/real = 2500
  train/fake = 2500
  val/real   = 313
  val/fake   = 313
  test/real  = 313
  test/fake  = 313
  train total = 5000
  val total   = 626
  test total  = 626
  all total   = 6252

抽樣後 race 分布：
  White  


輸出資料夾數量檢查：
  train/real:   2500
  train/fake:   2500
  val  /real:    313
  val  /fake:    313
  test /real:    313
  test /fake:    313
  TOTAL     :   6252

資料集建立完成。
